In [1]:
! pip3 install --upgrade --quiet --user google-cloud-aiplatform==1.88.0

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kfp 2.14.2 requires protobuf<7.0,==6.31.1, but you have protobuf 5.29.5 which is incompatible.
kfp-pipeline-spec 2.14.0 requires protobuf<7.0,==6.31.1, but you have protobuf 5.29.5 which is incompatible.


In [2]:
# Restart kernel after installs so that your environment can access the new packages
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

In [1]:
import vertexai

PROJECT_ID = ! gcloud config get-value project
PROJECT_ID = PROJECT_ID[0]
LOCATION = "us-central1"

print(PROJECT_ID)

vertexai.init(project=PROJECT_ID, location=LOCATION)

qwiklabs-gcp-00-c85b3fb112bb


In [2]:
import requests
from vertexai.generative_models import (
    Content,
    FunctionDeclaration,
    GenerationConfig,
    GenerativeModel,
    Part,
    Tool,
)

In [3]:
# --- Pure Python functions the model can call ---

def add(a: float, b: float) -> float:
    print("Calling add function")
    return a + b

def multiply(a: float, b: float) -> float:
    print("Calling multiply function")
    return a * b


# --- Function Declarations for tool-calling ---

add_decl = FunctionDeclaration(
    name="add",
    description="Add two numbers and return the sum.",
    parameters={
        "type": "OBJECT",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
    },
)

multiply_decl = FunctionDeclaration(
    name="multiply",
    description="Multiply two numbers and return the product.",
    parameters={
        "type": "OBJECT",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
    },
)

# Combine into a Tool
math_tool = Tool(function_declarations=[add_decl, multiply_decl])


In [4]:
system_instructions = (
    "Fulfill the user's instructions, including telling jokes. "
    "If asked to add or multiply numbers, call the provided functions. "
    "You may call one function after the other if needed. "
    "After receiving a function result, repeat the result back to the user."
)

model = GenerativeModel(
    model_name="gemini-2.0-flash-001",
    tools=[math_tool],
    generation_config=GenerationConfig(temperature=0),
    system_instruction=system_instructions,
)

chat = model.start_chat()


In [5]:
def handle_response(response):
    """
    If the model requests a function call, run it and send the function's result
    back to the chat. Otherwise, print the model's text.
    """
    # If there is a function call then invoke it; otherwise, print the reply.
    fcalls = getattr(response.candidates[0], "function_calls", None)
    if not fcalls:
        print(response.text)
        return

    function_call = fcalls[0]
    name = function_call.name
    args = function_call.args or {}

    if name == "add":
        a = args.get("a")
        b = args.get("b")
        result = add(a, b)

        tool_response = Content(
            role="tool",
            parts=[
                Part.from_function_response(
                    name="add",
                    response={"result": result},
                )
            ],
        )
        next_response = chat.send_message(tool_response)
        handle_response(next_response)

    elif name == "multiply":
        a = args.get("a")
        b = args.get("b")
        result = multiply(a, b)

        tool_response = Content(
            role="tool",
            parts=[
                Part.from_function_response(
                    name="multiply",
                    response={"result": result},
                )
            ],
        )
        next_response = chat.send_message(tool_response)
        handle_response(next_response)

    else:
        # You shouldn't end up here with the current toolset
        print("Unknown function call:", function_call)


In [8]:
response = chat.send_message("Tell me a joke?")
handle_response(response)


Why did the scarecrow win an award?

Because he was outstanding in his field!



In [7]:
response = chat.send_message("I have 7 pizzas each with 16 slices. How many slices do I have?")
handle_response(response)


Calling multiply function
You have 112 slices.



In [9]:
response = chat.send_message("Doug brought 3 pizzas. Andrew brought 4 pizzas. How many pizzas did they bring together?")
handle_response(response)


Calling add function
They brought 7 pizzas together.



In [10]:
response = chat.send_message("Doug brought 3 pizzas. Andrew brought 4 pizzas. There are 16 slices per pizza. How many slices are there?")
handle_response(response)


Calling add function
Calling multiply function
There are 112 slices.



In [11]:
response = chat.send_message("Doug brought 4 pizzas, but Andrew dropped 2 on the ground. How many pizzas are left?")
handle_response(response)


This question cannot be answered using the available tools.

